In [ ]:
# packages that needed to be installed
import pdfplumber
from PIL import Image
import camelot
import numpy as np
import pandas as pd
# packages that do not need to be installed
import os
import re
from collections import defaultdict
file_path = "MSAR002_-_MSAR_Premed_Course_Requirements_(1).pdf"

# --
def get_color_value(img_path): # extracts the color values from an image, averages it, then compares the red values 
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img)
    avg_color = arr.mean(axis=(0, 1)) 

    r, g, b = avg_color
    if abs(r - 242) <= 1.3: # red circle, 1.3 is meant for margin of error
        return -1  
    elif abs(r - 221) <= 1.3: # green circle
        return 1 
    elif abs(r - 247) <= 1.3:  # yellow circle
        return 0   
    else:
        return None  # unknown color
# -----

tables = camelot.read_pdf(file_path, pages="2") #2nd page, sets up our df
fixed_x = (551.61-527.85) # set width to not mess with the color values
os.makedirs("MSAR_Images/page2", exist_ok=True)
with pdfplumber.open(file_path) as pdf:
    page = pdf.pages[1]  
    images = page.images  # list of all images on the page
    for img in images:
        center_x = (img["x0"] + img["x1"]) / 2
        top, bottom = img["top"], img["bottom"]
        if (top > 70):
            x0_fixed = center_x - fixed_x / 2
            x1_fixed = center_x + fixed_x / 2
            cropped = page.crop((x0_fixed, top, x1_fixed, bottom)).to_image(resolution=200)
            cropped.save(f"MSAR_Images/page2/cell_image_{x0_fixed:.2f}_{top:.2f}.png") # made a new folder dedicated for the images

# extracting the colors
color_data = []
for file in os.listdir("MSAR_Images/page2"):
    if file.lower().endswith((".png")):
        file_path = os.path.join("MSAR_Images/page2", file)
        val = get_color_value(file_path)
        color_data.append((file, val))
groups = defaultdict(list)

for filename, value in color_data: # regex
    match = re.search(r"cell_image_(\d{3})\.\d+_", filename)
    if match:
        x_val = float(match.group(1))
        groups[x_val].append((value))
grouped_lists = list(groups.values())

# setting up our df
df = tables[0].df
df.columns = df.iloc[0]
df = df[1:]

rename_columns = {
    'Required or\nRecommended?': 'Required or Recommended?', 
    'Credit\nHours': 'Credit Hours', 
    'Pass\n/Fail' : 'Pass/Fail',
    'AP\nCredit' : 'AP Credit',
    'Online\nCourse' : 'Online Course',
    'Communit\nCollege' : 'Community College'
}

df = df.rename(columns=rename_columns)
df["Lab?"] = grouped_lists[0]
df["Pass/Fail"] = grouped_lists[1]
df["AP Credit"] = grouped_lists[2]
df["Online Course"] = grouped_lists[3]
df["Community College"] = grouped_lists[4]
df


C:\Users\georg\Downloads\DataBridge_Fall2025


,State,Medical School,Course,Class,Required or Recommended?,Additional Info,Credit Hours,Lab?,Pass/Fail,AP Credit,Online Course,Community College
1,AL,Frederick P. \nWhiddon College of \nMedicine a...,Anthropology,BESS,Required,Anthropology or any of the Behavioral and \nSo...,6,-1,0,0,1,1
2,,,Biochemistry,CHEM Recommended,,Biochemistry may take the place of Organic \nC...,,1,0,-1,0,1


In [ ]:
# we do it for all of the other pages now


column_names = ["State", "Medical School", "Course", "Class", 'Required or Recommended?', 'Additional Info', 'Credit Hours', 'Lab?', 'Pass/Fail', 'AP Credit', 'Online Course', 'Community College']
for i in range(213): # number of pages
    file_path = "MSAR002_-_MSAR_Premed_Course_Requirements_(1).pdf"
    tables = camelot.read_pdf(file_path, flavor = "lattice", pages=f"{i+3}") 

    fixed_x = (551.61-527.85) # set width to not mess with the color values
    os.makedirs(f"MSAR_Images/page{i + 3}", exist_ok=True)
    with pdfplumber.open(file_path) as pdf:
        page = pdf.pages[i + 2]  
        images = page.images  # list of all images on the page
        for img in images:
            center_x = (img["x0"] + img["x1"]) / 2
            top, bottom = img["top"], img["bottom"]
            if (top > 50):
                x0_fixed = center_x - fixed_x / 2
                x1_fixed = center_x + fixed_x / 2
                cropped = page.crop((x0_fixed, top, x1_fixed, bottom)).to_image(resolution=200)
                cropped.save(f"MSAR_Images/page{i + 3}/cell_image_{x0_fixed:.2f}_{top:.2f}.png") # made a new folder dedicated for the images
    # extracting the colors
    color_data = []
    for file in os.listdir(f"MSAR_Images/page{i + 3}"):
        if file.lower().endswith((".png")):
            file_path = os.path.join(f"MSAR_Images/page{i + 3}", file)
            val = get_color_value(file_path)
            color_data.append((file, val))
    groups = defaultdict(list)

    for filename, value in color_data:
        match = re.search(r"cell_image_(\d{3})\.\d+_", filename)
        if match:
            x_val = float(match.group(1))
            groups[x_val].append((value))
    grouped_lists = list(groups.values())

    # making a new dataframe
    dftemp = tables[0].df

    # filtering
    if (i+3) in [31, 56, 100, 115, 174]: # these specific page numbers does not have a header
        dftemp.columns = column_names
    else:
        dftemp.columns = dftemp.iloc[0]
        dftemp = dftemp.iloc[1:]
        dftemp = dftemp.rename(columns=rename_columns)
    if (len(dftemp.columns) == 13): # made an extra column lol
        dftemp = dftemp.drop(dftemp.columns[-1], axis = 1)
    if (dftemp["Course"].iloc[0] == ''): # accounting for pages that does not have the icons on the very first row
        dftemp = dftemp.iloc[1:]

    dftemp["Lab?"] = grouped_lists[0]
    dftemp["Pass/Fail"] = grouped_lists[1]
    dftemp["AP Credit"] = grouped_lists[2]
    dftemp["Online Course"] = grouped_lists[3]
    dftemp["Community College"] = grouped_lists[4]
    df = pd.concat([df, dftemp])
    df
    # took me around 15 minutes for it to run


C:\Users\georg\Downloads\DataBridge_Fall2025


C:\Users\georg\AppData\Local\Temp\ipykernel_75760\1475256470.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dftemp["Lab?"] = grouped_lists[0]
C:\Users\georg\AppData\Local\Temp\ipykernel_75760\1475256470.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dftemp["Pass/Fail"] = grouped_lists[1]
C:\Users\georg\AppData\Local\Temp\ipykernel_75760\1475256470.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

In [4]:
# cleaning up our dataframe

dftest = df
dftest = dftest.reset_index(drop = True)
for index, row in dftest.iterrows():
    if "\n" in row["Medical School"]:
        dftest.loc[index, "Medical School"] = dftest.loc[index, "Medical School"].replace("\n", "")
    if "\n" in row["Additional Info"]:
        dftest.loc[index, "Additional Info"] = dftest.loc[index, "Additional Info"].replace("\n", "")
    if dftest.loc[index, "Medical School"] == "":
        dftest.loc[index, "Medical School"] = dftest.loc[index - 1, "Medical School"]
        dftest.loc[index, "State"] = dftest.loc[index - 1, "State"]
    if row["Class"] == "CHEM Recommended":
        dftest.loc[index, "Class"] = "CHEM"
        dftest.loc[index, "Required or Recommended?"] = "Recommended"
    if row["Class"] == "CHEM Required":
        dftest.loc[index, "Class"] = "CHEM"
        dftest.loc[index, "Required or Recommended?"] = "Required"


dftest.to_csv("MSAR_table.csv", index = False)